In [3]:
# === FLAIR + mask shape catalog ==============================================
from pathlib import Path
import nibabel as nib

IMAGES_DIR = Path("/home/rbielski/SOOP/ds004889/acute_only/FLAIR_images/train")
MASKS_DIR = Path("/home/rbielski/SOOP/ds004889/acute_only/Acute_mask/train")

def _img_key(name: str) -> str:
    stem = name[:-7] if name.endswith(".nii.gz") else name
    return stem.replace("_FLAIR", "")

def _msk_key(name: str) -> str:
    s = name[:-7] if name.endswith(".nii.gz") else name
    return s.replace("_space-TRACE_desc-lesionAcute_mask", "")

images = { _img_key(p.name): p for p in sorted(IMAGES_DIR.glob("*.nii.gz")) if "mask" not in p.name }
masks  = { _msk_key(p.name): p for p in sorted(MASKS_DIR.glob("*.nii.gz")) if "mask" in p.name }

pairs = sorted(set(images) & set(masks))
catalog = []
for key in pairs:
    img_path = images[key]
    msk_path = masks[key]
    img_shape = nib.load(str(img_path)).shape[:3]
    msk_shape = nib.load(str(msk_path)).shape[:3]
    img_voxels = int(img_shape[0] * img_shape[1] * img_shape[2])
    catalog.append((key, img_path.name, img_shape, msk_shape, img_voxels))

catalog.sort(key=lambda entry: entry[-1], reverse=True)
print(f"Total paired cases: {len(catalog)}")
for idx, (_, name, img_shape, msk_shape, voxels) in enumerate(catalog, start=1):
    print(f"{idx:03d}. {name} -> image shape {img_shape} ({voxels:,} voxels), mask shape {msk_shape}")

Total paired cases: 680
001. sub-1003_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (480, 480, 25)
002. sub-1175_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (384, 384, 27)
003. sub-1270_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (384, 384, 25)
004. sub-139_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (384, 384, 26)
005. sub-1566_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (480, 480, 26)
006. sub-1587_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (384, 384, 25)
007. sub-1643_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (384, 384, 25)
008. sub-1660_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (384, 384, 24)
009. sub-313_FLAIR.nii.gz -> image shape (1024, 220, 1024) (230,686,720 voxels), mask shape (384, 384, 26)
010. s

# Goal: produce FLAIR volumes and masks that are pixel-wise aligned to your preprocessed DWI grid (same shape & affine) so you can train a segmentation model consistently across modalities.

## 1) Discover paired cases

### Scans the folders:

- FLAIR_DIR: raw FLAIR images.

- DWI_PP_DIR: preprocessed DWI outputs (*_img_prepped.nii.gz, *_mask_prepped.nii.gz).

- Extracts a sub-#### identifier from filenames and keeps the intersection of subjects present in both FLAIR and DWI-preprocessed sets.

- Prints how many subjects will be processed.

- Why: guarantees we only process subjects for which we already have the DWI preprocessed reference image and mask.

## 2) Load & canonicalize FLAIR (RAS+)

- Loads each raw FLAIR NIfTI and reorients it to canonical RAS+ using nibabel.

- Ensures data are truly 3-D (squeezes trailing 1-length dims), uses float32.

- Why: registration is more robust/consistent when inputs share a standard orientation.

## 3) Register FLAIR → DWI reference (fast multi-resolution)

- Chooses the preprocessed DWI image as the reference grid (target shape & affine).

- Tries a rigid (rotation/translation) registration with SimpleITK:

- Metric: Mattes Mutual Information (32 bins).

- Random sampling of voxels (default 10%).

- Multi-resolution pyramid (shrink factors e.g. [4,2,1]; smoothing [2,1,0]).

- Optimizer: Regular Step Gradient Descent with physical-unit scaling.

- Initialization: Centered (GEOMETRY) Euler3D.

- Time budget per case (e.g., REG_MAX_SEC = 25s). If the timer trips or anything errors → fallback.

- Fallback: if SITK is unavailable or registration exceeds time budget, it performs a shape-only resample (simple zoom to the reference shape; no spatial alignment beyond size).

### Why:

- MI + pyramid + sampling is much faster and generally sufficient for aligning contrasts (FLAIR vs DWI).

- The time budget prevents “hanging” on difficult subjects; you still get outputs via the safe fallback.

### Knobs you can tune:

- USE_SITK: turn registration on/off globally.

- REG_MAX_SEC: per-subject time limit.

- SHRINK, SMOOTH, SAMPLE_PCT, optimizer settings.

## 4) Resample and normalize on the DWI grid

### Resamples the FLAIR onto the exact DWI preprocessed grid:

- Same shape and affine as the DWI reference.

- Linear interpolation for the image.

- Mask selection: the mask used for saving is the preprocessed DWI mask (already aligned to the DWI grid), binarized to {0,1}.

- Intensity normalization (FLAIR):

- Compute on non-zero voxels only.

- Clip to [p1, p99], z-score inside the brain, then min-max to [0,1].

### Why: the model sees comparable intensity ranges and perfectly aligned inputs/labels.

## 5) Save with reference header (exact affine match)

### Uses the DWI preprocessed image header/affine when saving both the FLAIR and the mask:

### Outputs:

- ..._img_prepped.nii.gz (FLAIR in DWI space)

- ..._mask_prepped.nii.gz (DWI mask)

- This guarantees the FLAIR and mask affines are identical to the DWI preprocessed image.

### After each subject:

- Prints progress (subject index, quick registration status, file name).

- Frees memory with gc.collect().

### Why: identical affines + shapes remove all downstream alignment headaches (visualization, dataloaders, loss computation).

## 6) Post-run sanity & metadata

### After finishing, performs an assert check (for the last subject) that shapes and affines of:

- saved FLAIR, saved mask, and reference DWI image all match.

### Writes a preprocess_meta.json with:

- mode (FLAIR->DWI_space)

- subject count written

- whether SITK was used.

### Why: quick invariants and provenance help catch pipeline drift and ease debugging.

### Outputs & Guarantees

### For every common subject:

- *_img_prepped.nii.gz: FLAIR resampled into DWI preprocessed space.

- *_mask_prepped.nii.gz: the DWI preprocessed acute lesion mask (binary).

- Same shape and same affine across (FLAIR, DWI, mask) → ready for U-Net training with simple stacking or multi-modal loaders.

- Robust to missing SITK or stubborn registrations thanks to a deterministic fallback.

In [1]:
# ===================== Preprocess SOOP Acute (FLAIR -> DWI space) =====================
# - Pairs: strip "_FLAIR" from FLAIR; strip "_space-TRACE_desc-lesionAcute_mask" from masks
# - Reference: the *preprocessed DWI image+mask* (already target-shaped)
# - Steps:
#   * Load FLAIR, reorient to RAS+
#   * Rigid registration (SimpleITK, if available) FLAIR -> preprocessed DWI image
#   * Resample FLAIR onto DWI grid (same shape/affine)
#   * Normalize image; mask = the preprocessed DWI mask (already aligned)
#   * Save both using the DWI-preprocessed header so affines match exactly

from pathlib import Path
import os, json, logging, gc
import numpy as np
import nibabel as nib
from nibabel.orientations import axcodes2ornt, io_orientation, inv_ornt_aff, apply_orientation
import re

try:
    import SimpleITK as sitk
    _HAS_SITK = True
except Exception:
    _HAS_SITK = False

logging.getLogger("nibabel").setLevel(logging.ERROR)

# ---------------------------- PATHS -------------------------------------------
FLAIR_DIR     = Path("/home/rbielski/SOOP/ds004889/acute_only/FLAIR_images/train")
MASKS_DIR     = Path("/home/rbielski/SOOP/ds004889/acute_only/Acute_mask/train")  # raw masks (unused directly)
DWI_PP_DIR    = Path("/home/rbielski/SOOP/ds004889/acute_only/preprocessed/acute_train")  # <- your DWI preprocessed outputs
OUT_DIR       = Path("/home/rbielski/SOOP/ds004889/acute_only/preprocessed/flair_acute_train")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------ Pairing rules (SOOP naming) -------------------------
def _flair_key(name: str) -> str:
    s = name[:-7] if name.endswith(".nii.gz") else name
    return s.replace("_FLAIR", "")

def _mask_key(name: str) -> str:
    s = name[:-7] if name.endswith(".nii.gz") else name
    return s.replace("_space-TRACE_desc-lesionAcute_mask", "")

# Preprocessed DWI pairs (become our reference grid+affine)
def _pp_key_from_name(p: Path) -> str:
    return p.name.replace("_img_prepped.nii.gz", "").replace("_mask_prepped.nii.gz", "")

flair_raw = {p.name: p for p in FLAIR_DIR.glob("*.nii.gz") if "mask" not in p.name}
pp_dwi_imgs = {p.name: p for p in DWI_PP_DIR.glob("*_img_prepped.nii.gz")}
pp_dwi_msks = {p.name: p for p in DWI_PP_DIR.glob("*_mask_prepped.nii.gz")}

SID = re.compile(r"(sub-\d+)")

def flair_sid(name: str) -> str | None:
    m = SID.search(name)
    return m.group(1) if m else None

def dwi_sid_from_ppkey(name: str) -> str | None:
    # examples:
    #  sub-1000_space-TRACE_desc-lesionAcute_img_prepped.nii.gz -> sub-1000
    m = SID.search(name)
    return m.group(1) if m else None

# index by subject id
flair_by_sid = {}
for nm, p in flair_raw.items():
    sid = flair_sid(nm)
    if sid: flair_by_sid[sid] = p

dwi_by_sid = {}
for nm, p in pp_dwi_imgs.items():
    sid = dwi_sid_from_ppkey(nm)
    if sid: dwi_by_sid.setdefault(sid, {})["img_pp"] = p
for nm, p in pp_dwi_msks.items():
    sid = dwi_sid_from_ppkey(nm)
    if sid: dwi_by_sid.setdefault(sid, {})["msk_pp"] = p

common_sids = sorted(s for s in flair_by_sid.keys() if s in dwi_by_sid and "img_pp" in dwi_by_sid[s] and "msk_pp" in dwi_by_sid[s])
if not common_sids:
    raise RuntimeError("No overlapping subjects between FLAIR raw and preprocessed DWI pairs. "
                       "Check input folders and naming.")

print(f"Subjects in common: {len(common_sids)}")

def _ensure_3d(arr: np.ndarray) -> np.ndarray:
    v = np.asarray(arr)
    while v.ndim > 3 and v.shape[-1] == 1:
        v = v[..., 0]
    if v.ndim != 3:
        raise ValueError(f"Expected 3D or 4D with trailing 1, got {arr.shape}")
    return v.astype(np.float32)


# ----------------------- helpers ---------------------------------------------
def _to_ras(nii: nib.Nifti1Image) -> nib.Nifti1Image:
    RAS = axcodes2ornt(("R","A","S"))
    cur = io_orientation(nii.affine)
    data = _ensure_3d(nii.get_fdata())
    if np.allclose(cur, RAS):
        return nib.Nifti1Image(data, nii.affine, header=nii.header)
    xform = nib.orientations.ornt_transform(cur, RAS)
    data_ras = apply_orientation(data, xform)
    aff_ras  = nii.affine @ inv_ornt_aff(xform, nii.shape)
    return nib.Nifti1Image(data_ras, aff_ras, header=nii.header)


def _normalize(img: np.ndarray) -> np.ndarray:
    nz = img[img > 0]
    if nz.size == 0:
        return np.zeros_like(img, np.float32)
    p1, p99 = np.percentile(nz, [1, 99])
    img = np.clip(img, p1, p99)
    m, s = nz.mean(), nz.std()
    if s > 0: img = (img - m) / s
    mn, mx = img.min(), img.max()
    if mx > mn: img = (img - mn) / (mx - mn)
    return img.astype(np.float32)

def _sitk_from_nib(nii: nib.Nifti1Image) -> 'sitk.Image':
    import numpy as np, SimpleITK as sitk
    from nibabel.orientations import axcodes2ornt, io_orientation, ornt_transform, apply_orientation, inv_ornt_aff
    from nibabel.affines import voxel_sizes

    # Canonicalize to RAS+ so the affine rotation is clean
    RAS = axcodes2ornt(("R","A","S"))
    cur = io_orientation(nii.affine)
    data = nii.get_fdata().astype(np.float32)
    if not np.allclose(cur, RAS):
        xform = ornt_transform(cur, RAS)
        data = apply_orientation(data, xform)
        aff  = nii.affine @ inv_ornt_aff(xform, nii.shape)
    else:
        aff = nii.affine

    # Nibabel -> voxel sizes and rotation/origin (x,y,z)
    sx, sy, sz = [float(v) for v in voxel_sizes(aff)]
    R = aff[:3,:3] / np.array([sx, sy, sz], dtype=np.float32)
    # Orthonormalize rotation (robust to tiny numeric drift)
    U, _, Vt = np.linalg.svd(R, full_matrices=False)
    Rn = (U @ Vt).astype(np.float64)
    ox, oy, oz = [float(v) for v in aff[:3, 3]]

    # SimpleITK expects GetImageFromArray input as z,y,x
    sitk_img = sitk.GetImageFromArray(np.ascontiguousarray(data.transpose(2,1,0)))

    # *** IMPORTANT: set meta in X,Y,Z order ***
    sitk_img.SetSpacing((sx, sy, sz))      # not reversed
    sitk_img.SetOrigin((ox, oy, oz))       # not reversed
    sitk_img.SetDirection(Rn.flatten())    # row-major x,y,z
    return sitk_img




import time

# knobs
USE_SITK   = _HAS_SITK
REG_MAX_SEC_RIGID  = 25
REG_MAX_SEC_SPLINE = 20
SHRINK = [4, 2, 1]
SMOOTH = [2, 1, 0]
SAMPLE_PCT = 0.10

ENABLE_BSPLINE_REFINEMENT = True
BSP_GRID_SPACING_MM = (60.0, 60.0, 60.0)   # tune 40–80 if needed
BSP_MAX_ITERS = 50

def _shape_only_resample(ref_nii: nib.Nifti1Image, mov_nii: nib.Nifti1Image) -> nib.Nifti1Image:
    from scipy.ndimage import zoom
    mov_data = _ensure_3d(mov_nii.get_fdata())
    zf = tuple(r / i for r, i in zip(ref_nii.shape[:3], mov_data.shape))
    out = zoom(mov_data, zf, order=1, mode="nearest", prefilter=False).astype(np.float32)
    return nib.Nifti1Image(out, ref_nii.affine, header=ref_nii.header)

def _resample_like(ref_nii: nib.Nifti1Image, mov_nii: nib.Nifti1Image) -> nib.Nifti1Image:
    """
    Register mov_nii (FLAIR) to ref_nii (DWI-prep):
      1) Rigid MI registration (multi-res) with time-guard via StopRegistration().
      2) Optional coarse BSpline refinement (on rigid-resampled moving), with
         AbortOnMetricError and overlap-gating.
    Returns NIfTI on ref grid. If BSpline fails/skips, returns rigid result.
    """
    if not USE_SITK:
        return _shape_only_resample(ref_nii, mov_nii)

    try:
        # Build SITK images
        ref = _sitk_from_nib(ref_nii)   # fixed (DWI-prep)
        mov = _sitk_from_nib(mov_nii)   # moving (FLAIR)

        # ---- Stage 1: Rigid ----
        R = sitk.ImageRegistrationMethod()
        R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=32)
        R.SetMetricSamplingStrategy(R.RANDOM)
        R.SetMetricSamplingPercentage(float(SAMPLE_PCT))
        R.SetInterpolator(sitk.sitkLinear)
        R.SetOptimizerAsRegularStepGradientDescent(
            learningRate=1.0, minStep=1e-3, numberOfIterations=100,
            gradientMagnitudeTolerance=1e-6
        )
        R.SetOptimizerScalesFromPhysicalShift()
        R.SetShrinkFactorsPerLevel(SHRINK)
        R.SetSmoothingSigmasPerLevel(SMOOTH)
        R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        R.SetInitialTransform(
            sitk.CenteredTransformInitializer(
                ref, mov, sitk.Euler3DTransform(),
                sitk.CenteredTransformInitializerFilter.MOMENTS
            )
        )

        t0 = time.time()
        def _guard_rigid():
            if (time.time() - t0) > REG_MAX_SEC_RIGID:
                R.StopRegistration()  # graceful abort (no exception) :contentReference[oaicite:3]{index=3}
        R.AddCommand(sitk.sitkIterationEvent, _guard_rigid)

        print("   • rigid registration…", end="", flush=True)
        tx_rigid = R.Execute(ref, mov)
        print(" done", flush=True)

        # Rigid-resample moving for refinement & to have a rigid fallback
        mov_rigid = sitk.Resample(mov, ref, tx_rigid, sitk.sitkLinear, 0.0)
        rigid_out = sitk.GetArrayFromImage(mov_rigid).transpose(2,1,0).astype(np.float32)
        rigid_nib = nib.Nifti1Image(rigid_out, ref_nii.affine, header=ref_nii.header)

        if not ENABLE_BSPLINE_REFINEMENT:
            return rigid_nib

       # --- bspline refinement (optional) ---
        # gate by overlap after rigid
        fixed_mask  = sitk.OtsuThreshold(ref,       0, 1)
        moving_mask = sitk.OtsuThreshold(mov_rigid, 0, 1)
        ov = sitk.LabelOverlapMeasuresImageFilter(); ov.Execute(fixed_mask, moving_mask)
        if ov.GetJaccardCoefficient() < 0.50:
            print("   • bspline skipped (low overlap after rigid)")
            return rigid_nib

        # coarse grid (~80 mm), safer first pass
        phys = [sz*sp for sz, sp in zip(ref.GetSize(), ref.GetSpacing())]
        mesh = [max(1, int(s/100.0)-1) for s in phys]
        tx_bs = sitk.BSplineTransformInitializer(ref, transformDomainMeshSize=mesh, order=3)

        R2 = sitk.ImageRegistrationMethod()
        R2.SetMetricAsMattesMutualInformation(32)
        R2.SetMetricSamplingStrategy(R2.RANDOM)
        R2.SetMetricSamplingPercentage(0.10)
        R2.SetInterpolator(sitk.sitkLinear)
        R2.SetInitialTransform(tx_bs, inPlace=False)
        R2.SetShrinkFactorsPerLevel([2, 1])
        R2.SetSmoothingSigmasPerLevel([1.0, 0.0])
        R2.SetMetricFixedMask(fixed_mask)
        R2.SetMetricMovingMask(moving_mask)
        R2.SetOptimizerAsLBFGSB(gradientConvergenceTolerance=1e-5,
                                numberOfIterations=int(BSP_MAX_ITERS),
                                maximumNumberOfCorrections=5)

        # time guard -> graceful stop (no exceptions)
        t1 = time.time()
        def _guard_bs():
            if (time.time() - t1) > REG_MAX_SEC_SPLINE:
                R2.StopRegistration()  # supported early-exit
        R2.AddCommand(sitk.sitkIterationEvent, _guard_bs)

        print("   • bspline refinement…", end="", flush=True)
        try:
            tx_bs_fit = R2.Execute(ref, mov_rigid)
            print(" done", flush=True)
            tx_comp = sitk.CompositeTransform(ref.GetDimension())
            tx_comp.AddTransform(tx_rigid)
            tx_comp.AddTransform(tx_bs_fit)
            res = sitk.Resample(mov, ref, tx_comp, sitk.sitkLinear, 0.0)
            out_xyz = sitk.GetArrayFromImage(res).transpose(2,1,0).astype(np.float32)
            return nib.Nifti1Image(out_xyz, ref_nii.affine, header=ref_nii.header)
        except Exception as e_bs:
            print(f"   ⚠️ bspline failed → using rigid-only result ({e_bs})")
            return rigid_nib
    

        except Exception as e_bs:
            print(f"   ⚠️ bspline failed → using rigid-only result ({e_bs})")
            return rigid_nib

    except Exception as e:
        print(f"\n   ⚠️ registration failed → shape-only resample ({e})")
        return _shape_only_resample(ref_nii, mov_nii)



# ---------------------------- Main loop (with axis-order fix) -----------------
# ---------------------------- Main loop (with inline diagnostics) -----------------
from nibabel.orientations import io_orientation
from nibabel.affines import voxel_sizes

def _ornt_codes(aff):
    try:
        return "".join(nib.orientations.ornt2axcodes(io_orientation(aff)))
    except Exception:
        return "???"

def _extent_from_affine(aff: np.ndarray, shape_xyz: tuple[int,int,int]):
    """
    Rough physical extents along each axis using voxel sizes and translation.
    Returns ((x0,x1), (y0,y1), (z0,z1)).
    """
    sx, sy, sz = [float(v) for v in voxel_sizes(aff)]
    ox, oy, oz = [float(v) for v in aff[:3, 3]]
    nx, ny, nz = [int(v) for v in shape_xyz[:3]]
    return ((ox, ox + sx * (nx - 1)),
            (oy, oy + sy * (ny - 1)),
            (oz, oz + sz * (nz - 1)))

def _axis_nonzero_profile(arr_xyz: np.ndarray):
    """Counts how many index positions along each axis have any nonzero signal."""
    x_prof = (arr_xyz.sum(axis=(1,2)) > 0).sum()
    y_prof = (arr_xyz.sum(axis=(0,2)) > 0).sum()
    z_prof = (arr_xyz.sum(axis=(0,1)) > 0).sum()
    return x_prof, y_prof, z_prof

def _print_meta(prefix: str, nii: nib.Nifti1Image):
    sh = nii.shape[:3]
    vz = tuple(round(v, 6) for v in voxel_sizes(nii.affine))
    print(f"{prefix} shape={sh}, zooms={vz}, orient={_ornt_codes(nii.affine)}")
    ex = _extent_from_affine(nii.affine, sh)
    print(f"{prefix} extent X:{tuple(round(v,3) for v in ex[0])} "
          f"Y:{tuple(round(v,3) for v in ex[1])} Z:{tuple(round(v,3) for v in ex[2])}")

def _sitk_quick_meta(sitk_img):
    return dict(
        spacing=tuple(float(v) for v in sitk_img.GetSpacing()),
        origin =tuple(float(v) for v in sitk_img.GetOrigin()),
        direction=tuple(float(v) for v in sitk_img.GetDirection())
    )

written = 0
for sid in common_sids:
    try:
        flair_path = flair_by_sid[sid]
        ref_img_p  = dwi_by_sid[sid]["img_pp"]   # preprocessed DWI image (reference grid)
        ref_msk_p  = dwi_by_sid[sid]["msk_pp"]   # preprocessed DWI mask

        # use the *DWI preprocessed key* for saving so viewer aligns with DWI
        pp_key = ref_img_p.name.replace("_img_prepped.nii.gz", "")

        # load reference (DWI preprocessed) and FLAIR
        ref_img_nii = nib.load(str(ref_img_p))
        ref_msk_nii = nib.load(str(ref_msk_p))
        flair_nii   = nib.load(str(flair_path))
        flair_nii   = _to_ras(flair_nii)  # canonical orientation first

        print(f"\n=== {sid} :: diagnostics BEFORE registration/resample ===")
        _print_meta("REF(DWI_pp)", ref_img_nii)
        _print_meta("MOV(FLAIR_ras)", flair_nii)

        # --- optional: SITK-side quick meta (before registration) ---
        if USE_SITK and _HAS_SITK:
            ref_sitk = _sitk_from_nib(ref_img_nii)
            mov_sitk = _sitk_from_nib(flair_nii)
            print("SITK REF meta:", _sitk_quick_meta(ref_sitk))
            print("SITK MOV meta:", _sitk_quick_meta(mov_sitk))

        # 1) resample the FLAIR *physically* onto DWI grid
        flair_on_ref = _resample_like(ref_img_nii, flair_nii)

        # 2) no extra axis flips here; _resample_like returns x,y,z aligned to ref
        flair_arr = flair_on_ref.get_fdata().astype(np.float32)

        # --- diagnostics AFTER resample ---
        print(f"--- {sid} :: diagnostics AFTER resample ---")
        _print_meta("OUT(FLAIR_on_ref)", flair_on_ref)

        # nonzero profiles to detect “collapsed axis” problems early
        nx, ny, nz = _axis_nonzero_profile(flair_arr)
        sx, sy, sz = flair_arr.shape
        print(f"Nonzero index counts  X:{nx}/{sx}  Y:{ny}/{sy}  Z:{nz}/{sz}")
        assert min(nx, ny) > 0.05*max(sx, sy), \
            "⚠️ Nonzero support suggests collapse along X or Y (check SITK metadata order)."

        # 3) normalize; mask is already aligned with reference grid
        flair_arr = _normalize(flair_arr)
        mask_arr  = (ref_msk_nii.get_fdata().astype(np.float32) > 0.5).astype(np.float32)

        # 4) save using the *reference* header/affine so shapes+affines match bit-for-bit
        out_img = OUT_DIR / f"{pp_key}_img_prepped.nii.gz"
        out_msk = OUT_DIR / f"{pp_key}_mask_prepped.nii.gz"
        nib.save(nib.Nifti1Image(flair_arr, ref_img_nii.affine, header=ref_img_nii.header), str(out_img))
        nib.save(nib.Nifti1Image(mask_arr,  ref_img_nii.affine, header=ref_img_nii.header), str(out_msk))

        # 5) per-case sanity checks (shape, affine, and orientation)
        ppi = nib.load(str(out_img))
        ppm = nib.load(str(out_msk))
        assert ppi.shape == ref_img_nii.shape == ppm.shape, \
            f"shape mismatch: {ppi.shape} / {ref_img_nii.shape} / {ppm.shape}"
        assert np.allclose(ppi.affine, ref_img_nii.affine) and np.allclose(ppm.affine, ref_img_nii.affine), \
            "affine mismatch with reference"
        assert np.allclose(io_orientation(ppi.affine), io_orientation(ref_img_nii.affine)) and \
               np.allclose(io_orientation(ppm.affine), io_orientation(ref_img_nii.affine)), \
            "orientation mismatch with reference"

        written += 1
        print(f"✅ {sid} ok — saved {out_img.name}, {out_msk.name}")

    except Exception as e:
        print(f"⚠️  {sid}: {e}")
    finally:
        gc.collect()

# write meta once
meta = {"mode": "FLAIR->DWI_space", "subjects_written": written, "used_sitk": _HAS_SITK}
(OUT_DIR / "preprocess_meta.json").write_text(json.dumps(meta, indent=2))
print(f"\nDone. Wrote {written} FLAIR(+mask) pairs to {OUT_DIR}.")



Subjects in common: 680

=== sub-100 :: diagnostics BEFORE registration/resample ===
REF(DWI_pp) shape=(512, 512, 32), zooms=(0.898438, 0.898438, 5.999998), orient=LAS
REF(DWI_pp) extent X:(101.127, 560.228) Y:(-103.846, 355.255) Z:(-100.997, 85.003)
MOV(FLAIR_ras) shape=(336, 221, 336), zooms=(0.602677, 1.0, 0.602677), orient=RAS
MOV(FLAIR_ras) extent X:(-95.943, 105.953) Y:(-108.492, 111.508) Z:(-96.522, 105.375)
SITK REF meta: {'spacing': (0.89843752614038, 0.8984375089907367, 5.999997573287056), 'origin': (-351.58429062366486, -117.4084495306015, -25.8774733543396), 'direction': (0.986080639224037, 0.00758871908739081, 0.1660945040936855, 0.029540604716010443, 0.9750685831609904, -0.2199286493511731, -0.16362250951665264, 0.22177391522681705, 0.961272076419804)}
SITK MOV meta: {'spacing': (0.6026765523844967, 1.0000000373637843, 0.6026765180606658), 'origin': (-95.94343566894531, -108.49198913574219, -96.5219497680664), 'direction': (0.9972964004116462, -0.07323296331333154, 0.0060

In [ ]:
# ==== Preprocessed FLAIR/DWI viewer (axial only, raw+prep affine/voxel checks) ====
from pathlib import Path
import json, logging, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
from functools import lru_cache
from scipy.ndimage import binary_dilation

logging.getLogger("nibabel").setLevel(logging.ERROR)

# ---- Paths ----
RAW_FLAIR_DIR = Path("/home/rbielski/SOOP/ds004889/acute_only/FLAIR_images/train")
RAW_MASKS_DIR = Path("/home/rbielski/SOOP/ds004889/acute_only/Acute_mask/train")
PP_FLAIR_DIR  = Path("/home/rbielski/SOOP/ds004889/acute_only/preprocessed/flair_acute_train")
PP_DWI_DIR    = Path("/home/rbielski/SOOP/ds004889/acute_only/preprocessed/acute_train")  # optional (DWI prep)

# ---- Key rules (match your preprocess scripts) ----
def _flair_key(name: str) -> str:
    s = name[:-7] if name.endswith(".nii.gz") else name
    return s.replace("_FLAIR", "")

def _mask_key(name: str) -> str:
    s = name[:-7] if name.endswith(".nii.gz") else name
    return s.replace("_space-TRACE_desc-lesionAcute_mask", "")

def _pp_key_from_name(p: Path) -> str:
    return p.name.replace("_img_prepped.nii.gz", "").replace("_mask_prepped.nii.gz", "")

# ---- Collect preprocessed FLAIR+mask pairs ----
pp_imgs  = sorted(PP_FLAIR_DIR.glob("*_img_prepped.nii.gz"))
pp_masks = sorted(PP_FLAIR_DIR.glob("*_mask_prepped.nii.gz"))
PP = {}
for p in pp_imgs:
    PP.setdefault(_pp_key_from_name(p), {})["flair_img"] = p
for p in pp_masks:
    PP.setdefault(_pp_key_from_name(p), {})["mask"] = p
PP = {k: v for k, v in PP.items() if "flair_img" in v and "mask" in v}
if not PP:
    raise RuntimeError(f"No preprocessed FLAIR pairs found in {PP_FLAIR_DIR}")

# ---- Optionally collect preprocessed DWI for the same keys ----
PP_DWI = {}
if PP_DWI_DIR.exists():
    for dwi_p in PP_DWI_DIR.glob("*_img_prepped.nii.gz"):
        PP_DWI[_pp_key_from_name(dwi_p)] = dwi_p

# ---- Map raw FLAIR+mask for info readouts ----
raw_flair = {_flair_key(p.name): p for p in RAW_FLAIR_DIR.glob("*.nii.gz") if "mask" not in p.name}
raw_masks = {_mask_key(p.name): p for p in RAW_MASKS_DIR.glob("*.nii.gz")   if "mask" in p.name}
RAW = {}
for k in PP.keys():
    if k in raw_flair and k in raw_masks:
        RAW[k] = {"flair_img": raw_flair[k], "mask": raw_masks[k]}
    else:
        # fallback fuzzy match
        ri = next((p for kk, p in raw_flair.items() if kk in k or k in kk), None)
        rm = next((p for kk, p in raw_masks.items() if kk in k or k in kk), None)
        if ri and rm:
            RAW[k] = {"flair_img": ri, "mask": rm}

# ---- Target shape from meta (if present) ----
meta_path = PP_FLAIR_DIR / "preprocess_meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    TARGET_SHAPE = tuple(meta.get("target_shape", ()))
else:
    sample_img = next(iter(PP.values()))["flair_img"]
    TARGET_SHAPE = nib.load(str(sample_img)).shape[:3]

# ---- Helpers ----
@lru_cache(maxsize=128)
def _load_nii(path: str) -> np.ndarray:
    arr = nib.load(path).get_fdata()
    if arr.ndim == 4 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    return arr.astype(np.float32)

@lru_cache(maxsize=128)
def _load_img(path: str) -> nib.Nifti1Image:
    return nib.load(path)

def _normalize(img: np.ndarray) -> np.ndarray:
    if img.size == 0 or np.max(img) == 0:
        return np.zeros_like(img, dtype=np.float32)
    nz = img[img > 0]
    if nz.size == 0:
        return np.zeros_like(img, dtype=np.float32)
    p1, p99 = np.percentile(nz, [1, 99])
    img = np.clip(img, p1, p99)
    m, s = nz.mean(), nz.std()
    if s > 0:
        img = (img - m) / s
    mn, mx = img.min(), img.max()
    return ((img - mn) / (mx - mn + 1e-8)).astype(np.float32)

def _edges2d(mask2d: np.ndarray) -> np.ndarray:
    m = mask2d.astype(bool)
    return binary_dilation(m) & (~m)

def _zooms3(img: nib.Nifti1Image):
    z = img.header.get_zooms()
    return tuple(float(v) for v in z[:3])

def _affine_equal(a: np.ndarray, b: np.ndarray, tol=1e-4):
    return np.allclose(a, b, atol=tol)

# ---- UI ----
keys_sorted = sorted(PP.keys())
pair_dd   = W.Dropdown(options=keys_sorted, description="Case:", layout=W.Layout(width="100%"))
slice_sl  = W.IntSlider(description="Axial slice:", min=0, max=1, value=0, continuous_update=False, layout=W.Layout(width="60%"))
alpha_sl  = W.FloatSlider(description="Mask α:", min=0.0, max=1.0, step=0.05, value=0.55, layout=W.Layout(width="40%"))
edges_cb  = W.Checkbox(description="Edges only", value=True)
invert_cb = W.Checkbox(description="Invert image", value=False)
bg_opts   = ["FLAIR"]
# enable DWI background if we have a matching preprocessed DWI volume
if any(k in PP_DWI for k in keys_sorted):
    bg_opts.append("DWI")
bg_radio = W.RadioButtons(options=bg_opts, value=bg_opts[0], description="Background:", layout=W.Layout(width="30%"))

status = W.HTML(f"<b>Viewer</b> — cases: {len(keys_sorted)} | target: {TARGET_SHAPE}")
controls = W.VBox([
    status,
    pair_dd,
    W.HBox([slice_sl, alpha_sl]),
    W.HBox([edges_cb, invert_cb, bg_radio]),
])
out = W.Output()

def _update_slider_range(*_):
    key = pair_dd.value
    # choose volume for depth based on current background selection
    if bg_radio.value == "DWI" and key in PP_DWI:
        vol = _load_nii(str(PP_DWI[key]))
    else:
        vol = _load_nii(str(PP[key]["flair_img"]))
    slice_sl.max = int(vol.shape[2] - 1)
    slice_sl.value = min(slice_sl.value, slice_sl.max)

def _draw(*_):
    with out:
        clear_output(wait=True)
        try:
            key = pair_dd.value
            flair_pp_path = PP[key]["flair_img"]
            mask_pp_path  = PP[key]["mask"]
            flair_pp = _load_nii(str(flair_pp_path))
            mask_pp  = _load_nii(str(mask_pp_path)) > 0.5

            # background selection: FLAIR (default) or DWI (if available)
            if bg_radio.value == "DWI" and key in PP_DWI:
                bg_path = PP_DWI[key]
                bg_vol  = _load_nii(str(bg_path))
                bg_hdr  = _load_img(str(bg_path))
                bg_name = "DWI (preprocessed)"
            else:
                bg_path = flair_pp_path
                bg_vol  = flair_pp
                bg_hdr  = _load_img(str(flair_pp_path))
                bg_name = "FLAIR (preprocessed)"

            img_view = _normalize(bg_vol.copy())
            if invert_cb.value:
                img_view = 1.0 - img_view

            idx = int(slice_sl.value)
            img2d = img_view[:, :, idx]
            m2d   = mask_pp[:, :, idx]

            # status text: shapes, zooms
            raw_info = ""
            if key in RAW:
                ri = _load_img(str(RAW[key]["flair_img"]))
                rm = _load_img(str(RAW[key]["mask"]))
                raw_info = f" | raw(img/msk): {ri.shape[:3]}/{rm.shape[:3]}, zooms: {tuple(round(v,3) for v in _zooms3(ri))}/{tuple(round(v,3) for v in _zooms3(rm))}"

            ppi = _load_img(str(flair_pp_path))
            ppm = _load_img(str(mask_pp_path))
            prep_info = (f"| prep shape: {bg_hdr.shape[:3]}, zooms: {tuple(round(v,3) for v in _zooms3(bg_hdr))} "
                         f"| affines match (prep flair vs mask): "
                         f"{'✅' if _affine_equal(ppi.affine, ppm.affine) else '⚠️'}")
            if key in PP_DWI and bg_radio.value == "DWI":
                # also report FLAIR vs DWI preprocessed affine equality
                ppd = _load_img(str(PP_DWI[key]))
                prep_info += f" | affines match (prep DWI vs mask): {'✅' if _affine_equal(ppd.affine, ppm.affine) else '⚠️'}"

            status.value = (f"<b>Viewer</b> — cases: {len(keys_sorted)} {prep_info} {raw_info} "
                            f"| BG: <b>{bg_name}</b>")

            # draw
            plt.figure(figsize=(5.6, 5.6))
            plt.imshow(img2d.T, cmap="gray", origin="lower")
            if edges_cb.value:
                plt.contour(_edges2d(m2d).T, levels=[0.5], linewidths=0.7, colors="r")
            else:
                plt.imshow(np.ma.masked_where(~m2d.T, m2d.T), cmap="jet", alpha=float(alpha_sl.value), origin="lower")
            plt.axis("off"); plt.tight_layout(); plt.show(); plt.close()

            # print affine diagnostics
            if key in RAW:
                ri = _load_img(str(RAW[key]["flair_img"]))
                rm = _load_img(str(RAW[key]["mask"]))
                print("Raw FLAIR affine:\n", ri.affine)
                print("Raw Mask  affine:\n", rm.affine)
                print("Raw affines equal:", _affine_equal(ri.affine, rm.affine))
            print("\nPreprocessed FLAIR affine:\n", ppi.affine)
            print("Preprocessed Mask  affine:\n",  ppm.affine)
            print("Prep affines equal (FLAIR vs mask):", _affine_equal(ppi.affine, ppm.affine))
            if key in PP_DWI:
                ppd = _load_img(str(PP_DWI[key]))
                print("\nPreprocessed DWI affine:\n", ppd.affine)
                print("Prep affines equal (DWI vs mask):", _affine_equal(ppd.affine, ppm.affine))

        except Exception as exc:
            print("Draw error:", exc)

def _sync_depth_on_bg_change(*_):
    # keep slice index valid when switching BG modality
    _update_slider_range()
    _draw()

pair_dd.observe(_update_slider_range, names="value")
pair_dd.observe(_draw, names="value")
slice_sl.observe(_draw, names="value")
alpha_sl.observe(_draw, names="value")
edges_cb.observe(_draw, names="value")
invert_cb.observe(_draw, names="value")
bg_radio.observe(_sync_depth_on_bg_change, names="value")

_update_slider_range()
_draw()
display(controls, out)
# =============================================================================


Output()